In [10]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection

csv_path = project_root + "/data/air_traffic_gold.csv"
con = get_ibis_connection(
    backend="duckdb",
    duckdb_csv_path=csv_path,
)


In [11]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

llm = ChatOpenAI(base_url=base_url, api_key=api_key, temperature=0, model=model)


In [12]:
question = """
  How many passengers landed at terminal 1?
"""

In [13]:
llm_response = """
SELECT * 
  FROM air_traffic
WHERE
  'Terminal' = "Terminal 1"

"""

In [14]:
def test_query(con, query):
    error_message = None

    try:
        print(con.sql(query).execute())
    except Exception as e:
        print("Could not execute the query")
        error_message = e
        print(e)
    return error_message



In [15]:
error_message = test_query(con, llm_response)

Could not execute the query
Binder Error: Referenced column "Terminal 1" not found in FROM clause!
Candidate bindings: "Terminal", "Operating Airline", "Year", "Operating Airline IATA Code", "Passenger Count"

LINE 5:   'Terminal' = "Terminal 1"
                       ^


In [16]:
system = """
      Your role is to debug SQL error executed in Python via the ibis library
      The user question:
      {question}
      Here is the return query:
      {query}
      It returns the following error message:
      {error_message}
      Please make sure the response is clean and without and explanations
      Please don't use markdown format

  """.strip()

print(system)

Your role is to debug SQL error executed in Python via the ibis library
      The user question:
      {question}
      Here is the return query:
      {query}
      It returns the following error message:
      {error_message}
      Please make sure the response is clean and without and explanations
      Please don't use markdown format


In [17]:
user = "Please debug the error and returned a the correct SQL query"

In [18]:
from langchain_core.prompts import ChatPromptTemplate

messages = [("system", system), ("user", user)]

prompt = ChatPromptTemplate.from_messages(messages)


In [19]:
chain = prompt | llm


In [20]:
llm_response = chain.invoke({
  "question": question,
  "query": llm_response,
  "error_message": error_message
})

In [21]:
fixed_query = llm_response.content
print(fixed_query)

SELECT *
  FROM air_traffic
WHERE
  Terminal = 'Terminal 1'


In [22]:
error_message = test_query(con, fixed_query)


      Unnamed: 0  Year        Date Operating Airline  \
0              0  1999  1999-07-01      ATA Airlines   
1              1  1999  1999-07-01      ATA Airlines   
2              2  1999  1999-07-01      ATA Airlines   
3              5  1999  1999-07-01        Air Canada   
4              6  1999  1999-07-01        Air Canada   
...          ...   ...         ...               ...   
6996       39158  2025  2025-11-01  SkyWest Airlines   
6997       39159  2025  2025-11-01  SkyWest Airlines   
6998       39160  2025  2025-11-01  SkyWest Airlines   
6999       39161  2025  2025-11-01  SkyWest Airlines   
7000       39162  2025  2025-11-01  SkyWest Airlines   

     Operating Airline IATA Code  Published Airline  \
0                             TZ       ATA Airlines   
1                             TZ       ATA Airlines   
2                             TZ       ATA Airlines   
3                             AC         Air Canada   
4                             AC         Air Canada 